In [1]:
# Import libraries and StackSats classes needed for exporting strategy weights, merging data, and plotting results.
import sys
import polars as pl
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
import pandas as pd

from stacksats.runner.core import StrategyRunner
from stacksats.strategy_types import ExportConfig
from stacksats.strategies.stable.baselines.uniform import UniformStrategy
from stacksats.strategies.stable.signals.momentum import MomentumStrategy

_root = Path.cwd()
while not (_root / "src").exists():
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from src import config, data_utils, plots, strategy_utils

STACKSATS_DATA_PATH = config.STACKSATS_DATA_PATH
RAW_PATH = config.RAW_PATH
total_budget_usd = config.TOTAL_BUDGET_USD

data_utils.check_stacksats_data(STACKSATS_DATA_PATH, RAW_PATH)

True

In [2]:
# Initialize the runner and load the prepared Bitcoin analytics parquet manually.
runner = StrategyRunner()

btc_df = (
    pl.read_parquet(STACKSATS_DATA_PATH)
    .with_columns(pl.col("date").cast(pl.Datetime))
    .sort("date")
)

print("Loaded rows:", btc_df.height)

print(
    btc_df.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Loaded rows: 5689
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [3]:
# Restrict the data to the requested overall horizon while respecting actual available coverage.
btc_train = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2010, 8, 16)) &
        (pl.col("date") <= pl.datetime(2023, 12, 31)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Filtered rows:", btc_train.height)

print(
    btc_train.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Filtered rows: 4886
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-08-16 00:00:00 ┆ 2023-12-31 00:00:00 │
└─────────────────────┴─────────────────────┘


In [4]:
momentum_strategy = MomentumStrategy()
uniform_strategy = UniformStrategy()

momentum_yearly = []
uniform_yearly = []

for year in range(2018, 2024):
    momentum_result = strategy_utils.export_one_year(momentum_strategy, btc_train, year, runner)
    if momentum_result is not None:
        momentum_yearly.append(momentum_result)

    uniform_result = strategy_utils.export_one_year(uniform_strategy, btc_train, year, runner)
    if uniform_result is not None:
        uniform_yearly.append(uniform_result)

print("Momentum valid yearly exports:", len(momentum_yearly))
print("Uniform valid yearly exports:", len(uniform_yearly))

2018: exported 365 rows
2018: exported 365 rows
2019: exported 365 rows
2019: exported 365 rows
2020: exported 365 rows
2020: exported 365 rows
2021: exported 365 rows
2021: exported 365 rows
2022: exported 365 rows
2022: exported 365 rows
2023: exported 365 rows
2023: exported 365 rows
Momentum valid yearly exports: 6
Uniform valid yearly exports: 6


In [5]:
if not momentum_yearly:
    raise ValueError("No valid Momentum exports were produced.")

if not uniform_yearly:
    raise ValueError("No valid Uniform exports were produced.")

momentum_all = pl.concat(momentum_yearly).rename({"weight": "momentum_weight_raw"})
uniform_all = pl.concat(uniform_yearly).rename({"weight": "baseline_weight_raw"})

print("Momentum combined rows:", momentum_all.height)
print("Uniform combined rows:", uniform_all.height)

Momentum combined rows: 2190
Uniform combined rows: 2190


In [6]:
merged = (
    momentum_all
    .select(["date", "price_usd", "momentum_weight_raw"])
    .join(
        uniform_all.select(["date", "baseline_weight_raw"]),
        on="date",
        how="inner"
    )
    .sort("date")
)

print("Merged rows:", merged.height)

merged.head()

Merged rows: 2190


date,price_usd,momentum_weight_raw,baseline_weight_raw
datetime[μs],f64,f64,f64
2018-01-01 00:00:00,13466.31,0.00274,0.00274
2018-01-02 00:00:00,14888.11,0.00274,0.00274
2018-01-03 00:00:00,15098.14,0.00274,0.00274
2018-01-04 00:00:00,15144.99,0.00274,0.00274
2018-01-05 00:00:00,16960.01,0.00274,0.00274


In [7]:
# momentum_sum = merged["momentum_weight_raw"].sum()
# baseline_sum = merged["baseline_weight_raw"].sum()

# merged = merged.with_columns([
#     (pl.col("momentum_weight_raw") / momentum_sum).alias("momentum_weight"),
#     (pl.col("baseline_weight_raw") / baseline_sum).alias("baseline_weight"),
# ])

merged = merged.with_columns(pl.col("date").dt.year().alias("year"))

merged = merged.with_columns([
    (pl.col("momentum_weight_raw") / pl.col("momentum_weight_raw").sum().over("year")).alias("momentum_weight"),
    (pl.col("baseline_weight_raw") / pl.col("baseline_weight_raw").sum().over("year")).alias("baseline_weight"),
])

print("Momentum normalized weight sum:", merged["momentum_weight"].sum())
print("Baseline normalized weight sum:", merged["baseline_weight"].sum())

Momentum normalized weight sum: 6.000000000000001
Baseline normalized weight sum: 6.0


In [8]:
merged = merged.with_columns([
    (pl.col("momentum_weight") * total_budget_usd).alias("dynamic_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
])

merged = merged.with_columns([
    (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("btc_accum_dynamic") * 100_000_000).alias("sats_accum_dynamic"),
    (pl.col("btc_accum_baseline") * 100_000_000).alias("sats_accum_baseline"),
])

merged = merged.with_columns([
    (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
    (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
])

total_dynamic_btc = merged["btc_accum_dynamic"].sum()
total_baseline_btc = merged["btc_accum_baseline"].sum()

num_years = merged["date"].dt.year().n_unique()
total_invested = total_budget_usd * num_years

sats_per_dollar_dynamic = (total_dynamic_btc / total_invested) * 100_000_000
sats_per_dollar_baseline = (total_baseline_btc / total_invested) * 100_000_000
# sats_per_dollar_dynamic = (total_dynamic_btc / total_budget_usd) * 100_000_000
# sats_per_dollar_baseline = (total_baseline_btc / total_budget_usd) * 100_000_000

pct_diff_vs_baseline = (
    (total_dynamic_btc - total_baseline_btc) / total_baseline_btc
) * 100

performance_label = "better" if pct_diff_vs_baseline > 0 else "worse"

print(f"Total BTC accumulated, Momentum: {total_dynamic_btc:.6f}")
print(f"Total BTC accumulated, DCA: {total_baseline_btc:.6f}")
print(f"Sats per dollar, Momentum: {sats_per_dollar_dynamic:.2f}")
print(f"Sats per dollar, DCA: {sats_per_dollar_baseline:.2f}")
print(f"Momentum performed {abs(pct_diff_vs_baseline):.2f}% {performance_label} than DCA")

Total BTC accumulated, Momentum: 0.505604
Total BTC accumulated, DCA: 0.505128
Sats per dollar, Momentum: 8426.73
Sats per dollar, DCA: 8418.79
Momentum performed 0.09% better than DCA


In [9]:
plot_df = merged.to_pandas()
plot_df["date"] = pd.to_datetime(plot_df["date"])
plot_df["year"] = plot_df["date"].dt.year
plot_df["cycle_label"] = plot_df["date"].apply(plots.assign_cycle_label)

plot_df.head()

,date,price_usd,momentum_weight_raw,baseline_weight_raw,year,momentum_weight,baseline_weight,dynamic_usd,baseline_usd,btc_accum_dynamic,btc_accum_baseline,sats_accum_dynamic,sats_accum_baseline,sats_per_dollar_dynamic,sats_per_dollar_baseline,cycle_label
0,2018-01-01,13466.31,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000203,0.000203,20345.039045,20345.039045,7425.939251,7425.939251,Cycle 3: 2018-2021
1,2018-01-02,14888.11,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000184,0.000184,18402.107638,18402.107638,6716.769288,6716.769288,Cycle 3: 2018-2021
2,2018-01-03,15098.14,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000181,0.000181,18146.116193,18146.116193,6623.332410,6623.332410,Cycle 3: 2018-2021
3,2018-01-04,15144.99,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000181,0.000181,18089.982413,18089.982413,6602.843581,6602.843581,Cycle 3: 2018-2021
4,2018-01-05,16960.01,0.00274,0.00274,2018,0.00274,0.00274,2.739726,2.739726,0.000162,0.000162,16154.035448,16154.035448,5896.222939,5896.222939,Cycle 3: 2018-2021


In [10]:
top_buy_points_list = []

for year, year_df in plot_df.groupby("year"):
    threshold = year_df["momentum_weight"].quantile(0.90)

    top_year_df = year_df[year_df["momentum_weight"] >= threshold].copy()
    top_year_df["top_buy_threshold_year"] = threshold

    top_buy_points_list.append(top_year_df)

top_buy_points = pd.concat(top_buy_points_list, ignore_index=True)

top_buy_points = top_buy_points.sort_values(
    ["year", "momentum_weight"],
    ascending=[True, False]
)

print("Top buy points:", len(top_buy_points))
top_buy_points[["date", "year", "price_usd", "momentum_weight", "top_buy_threshold_year"]].head()

Top buy points: 222


,date,year,price_usd,momentum_weight,top_buy_threshold_year
4,2018-02-05,2018,6935.78,0.004553,0.003566
3,2018-02-04,2018,8204.70,0.004290,0.003566
5,2018-02-06,2018,7819.86,0.004150,0.003566
26,2018-12-07,2018,3361.62,0.004084,0.003566
32,2018-12-13,2018,3264.66,0.004037,0.003566


## Adding Test data

In [11]:
# Testing window: 2024 onwards (held-out period)
btc_test = (
    btc_df
    .filter(
        (pl.col("date") >= pl.datetime(2024, 1, 1)) &
        pl.col("price_usd").is_not_null()
    )
    .sort("date")
)

print("Test rows:", btc_test.height)
print(
    btc_test.select(
        pl.col("date").min().alias("min_date"),
        pl.col("date").max().alias("max_date")
    )
)

Test rows: 803
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ min_date            ┆ max_date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2024-01-01 00:00:00 ┆ 2026-03-13 00:00:00 │
└─────────────────────┴─────────────────────┘


In [12]:
momentum_test_yearly = []
uniform_test_yearly = []

test_years = sorted(btc_test["date"].dt.year().unique().to_list())
print("Test years available:", test_years)

for year in test_years:
    momentum_result = strategy_utils.export_one_year(momentum_strategy, btc_test, year, runner)
    if momentum_result is not None:
        momentum_test_yearly.append(momentum_result)

    uniform_result = strategy_utils.export_one_year(uniform_strategy, btc_test, year, runner)
    if uniform_result is not None:
        uniform_test_yearly.append(uniform_result)

print("Momentum test exports:", len(momentum_test_yearly))
print("Uniform test exports:", len(uniform_test_yearly))

Test years available: [2024, 2025, 2026]
2024: exported 365 rows
2024: exported 365 rows
2025: exported 365 rows
2025: exported 365 rows
2026: skipped, less than 365 rows
2026: skipped, less than 365 rows
Momentum test exports: 2
Uniform test exports: 2


In [13]:
# Build test merged dataframe (same pipeline as train)
momentum_test_all = pl.concat(momentum_test_yearly).rename({"weight": "momentum_weight_raw"})
uniform_test_all = pl.concat(uniform_test_yearly).rename({"weight": "baseline_weight_raw"})

merged_test = (
    momentum_test_all
    .select(["date", "price_usd", "momentum_weight_raw"])
    .join(uniform_test_all.select(["date", "baseline_weight_raw"]), on="date", how="inner")
    .sort("date")
)

merged_test = merged_test.with_columns(pl.col("date").dt.year().alias("year"))
merged_test = merged_test.with_columns([
    (pl.col("momentum_weight_raw") / pl.col("momentum_weight_raw").sum().over("year")).alias("momentum_weight"),
    (pl.col("baseline_weight_raw") / pl.col("baseline_weight_raw").sum().over("year")).alias("baseline_weight"),
])
merged_test = merged_test.with_columns([
    (pl.col("momentum_weight") * total_budget_usd).alias("dynamic_usd"),
    (pl.col("baseline_weight") * total_budget_usd).alias("baseline_usd"),
])
merged_test = merged_test.with_columns([
    (pl.col("dynamic_usd") / pl.col("price_usd")).alias("btc_accum_dynamic"),
    (pl.col("baseline_usd") / pl.col("price_usd")).alias("btc_accum_baseline"),
])
merged_test = merged_test.with_columns([
    (pl.col("btc_accum_dynamic") * 1e8).alias("sats_accum_dynamic"),
    (pl.col("btc_accum_baseline") * 1e8).alias("sats_accum_baseline"),
])
merged_test = merged_test.with_columns([
    (pl.col("sats_accum_dynamic") / pl.col("dynamic_usd")).alias("sats_per_dollar_dynamic"),
    (pl.col("sats_accum_baseline") / pl.col("baseline_usd")).alias("sats_per_dollar_baseline"),
])

test_plot_df = merged_test.to_pandas()
test_plot_df["date"] = pd.to_datetime(test_plot_df["date"])
test_plot_df["year"] = test_plot_df["date"].dt.year
test_plot_df["cycle_label"] = test_plot_df["date"].apply(plots.assign_cycle_label)

In [14]:
combined_plot_df = pd.concat([plot_df, test_plot_df]).sort_values("date").reset_index(drop=True)

cols = plots.StrategyColumns(
    weight="momentum_weight",
    spd="sats_per_dollar_dynamic",
    sats_accum="sats_accum_dynamic",
)

# Full period
full_plot = plots.plot_strategy_full_period(
    combined_plot_df, cols, 
    "Momentum", 
    date_range=("2018-01-01", str(combined_plot_df["date"].max().date())),
    test_start_date="2024-01-01"
)
plt.show()

In [15]:
top_buy_points_table = top_buy_points[
    [
        "date",
        "year",
        "cycle_label",
        "price_usd",
        "momentum_weight",
        "top_buy_threshold_year",
    ]
].copy()

top_buy_points_table["date"] = top_buy_points_table["date"].dt.strftime("%Y-%m-%d")
top_buy_points_table["price_usd"] = top_buy_points_table["price_usd"].round(2)
top_buy_points_table["momentum_weight"] = top_buy_points_table["momentum_weight"].round(8)
top_buy_points_table["top_buy_threshold_year"] = top_buy_points_table["top_buy_threshold_year"].round(8)

top_buy_points_table.head(50)

,date,year,cycle_label,price_usd,momentum_weight,top_buy_threshold_year
4,2018-02-05,2018,Cycle 3: 2018-2021,6935.78,0.004553,0.003566
3,2018-02-04,2018,Cycle 3: 2018-2021,8204.70,0.004290,0.003566
5,2018-02-06,2018,Cycle 3: 2018-2021,7819.86,0.004150,0.003566
26,2018-12-07,2018,Cycle 3: 2018-2021,3361.62,0.004084,0.003566
32,2018-12-13,2018,Cycle 3: 2018-2021,3264.66,0.004037,0.003566
27,2018-12-08,2018,Cycle 3: 2018-2021,3400.00,0.004023,0.003566
25,2018-12-06,2018,Cycle 3: 2018-2021,3445.00,0.004017,0.003566
30,2018-12-11,2018,Cycle 3: 2018-2021,3346.22,0.004013,0.003566
1,2018-02-02,2018,Cycle 3: 2018-2021,8779.44,0.004007,0.003566
6,2018-02-07,2018,Cycle 3: 2018-2021,7657.00,0.003990,0.003566
